# GS-RVFL Visualization Gallery

This notebook provides visualizations of GS-RVFL behavior and performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from gs_rvfl import GSRVFLClassifier
from gs_rvfl.operators import StructuredDirectLink, AdaptiveBias

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

%matplotlib inline

## 1. Decision Boundary Visualization

Visualize GS-RVFL decision boundaries on 2D datasets.

In [ ]:
def plot_decision_boundary(model, X, y, title='Decision Boundary'):
    """Plot decision boundary of a classifier."""
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predict on mesh grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=50)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    plt.colorbar(scatter)
    plt.show()

# Generate datasets
datasets = [
    ('Moons', make_moons(n_samples=200, noise=0.15, random_state=42)),
    ('Circles', make_circles(n_samples=200, noise=0.1, factor=0.5, random_state=42)),
    ('Blobs', make_classification(n_samples=200, n_features=2, n_classes=2,
                                   n_redundant=0, n_informative=2,
                                   random_state=42, n_clusters_per_class=1))
]

for name, (X, y) in datasets:
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )
    
    # Train model
    model = GSRVFLClassifier(n_hidden=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Plot
    plot_decision_boundary(model, X, y, f'GS-RVFL: {name} Dataset')

## 2. Operator Impact Visualization

Visualize how different operators affect the decision boundary.

In [ ]:
def plot_operator_comparison(X, y, title='Operator Comparison'):
    """Compare decision boundaries with different operators."""
    configs = [
        ('Standard RVFL', [], []),
        ('+ Structured DL', [StructuredDirectLink()], []),
        ('+ Adaptive Bias', [], [AdaptiveBias()]),
        ('GS-RVFL (Full)', [StructuredDirectLink()], [AdaptiveBias()])
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.ravel()
    
    for idx, (name, g_ops, b_ops) in enumerate(configs):
        # Train model
        model = GSRVFLClassifier(
            n_hidden=50,
            random_state=42,
            g_operators=g_ops,
            b_operators=b_ops
        )
        model.fit(X, y)
        
        # Plot decision boundary
        x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
        y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                             np.linspace(y_min, y_max, 100))
        
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        
        axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
        axes[idx].scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=30)
        axes[idx].set_title(f'{name}\nAccuracy: {accuracy_score(y, model.predict(X)):.3f}')
        axes[idx].set_xlabel('Feature 1')
        axes[idx].set_ylabel('Feature 2')
    
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Generate data
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
plot_operator_comparison(X, y, 'Impact of Different Operators on Decision Boundary')

## 3. Performance vs. Number of Hidden Nodes

In [ ]:
# Generate data
X, y = make_classification(n_samples=500, n_features=10, n_classes=2,
                           n_informative=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Test different numbers of hidden nodes
hidden_nodes = [10, 20, 50, 100, 200, 500, 1000]
accuracies = []
train_times = []

for n in hidden_nodes:
    model = GSRVFLClassifier(n_hidden=n, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))
    train_times.append(model._train_time if hasattr(model, '_train_time') else 0)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(hidden_nodes, accuracies, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Hidden Nodes')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Accuracy vs. Number of Hidden Nodes')
ax1.grid(True)
ax1.set_xscale('log')

# Train time (if available)
if any(train_times):
    ax2.plot(hidden_nodes, train_times, 'ro-', linewidth=2, markersize=8)
    ax2.set_xlabel('Number of Hidden Nodes')
    ax2.set_ylabel('Training Time (s)')
    ax2.set_title('Training Time vs. Number of Hidden Nodes')
    ax2.grid(True)
    ax2.set_xscale('log')
    ax2.set_yscale('log')
else:
    ax2.text(0.5, 0.5, 'Training time not recorded', 
             transform=ax2.transAxes, ha='center', va='center')

plt.tight_layout()
plt.show()

## 4. Feature Importance Visualization

Visualize the importance of different components.

In [ ]:
def plot_component_weights(model):
    """Plot weights for different components."""
    if not model._fitted:
        return
    
    beta = model.beta
    n_hidden = model.n_hidden
    n_g = model.n_g_features
    n_b = model.n_b_features
    
    # Split weights
    weights_h = beta[:n_hidden, :]
    weights_g = beta[n_hidden:n_hidden+n_g, :]
    weights_b = beta[n_hidden+n_g:n_hidden+n_g+n_b, :]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Hidden weights
    if weights_h.size > 0:
        axes[0].hist(np.abs(weights_h.flatten()), bins=30, alpha=0.7)
        axes[0].set_title(f'Hidden Weights (n={n_hidden})')
        axes[0].set_xlabel('|Weight|')
        axes[0].set_ylabel('Frequency')
    else:
        axes[0].text(0.5, 0.5, 'No hidden weights', ha='center', va='center')
    
    # G weights
    if weights_g.size > 0:
        axes[1].hist(np.abs(weights_g.flatten()), bins=30, alpha=0.7, color='green')
        axes[1].set_title(f'Structured Direct-Link Weights (n={n_g})')
        axes[1].set_xlabel('|Weight|')
        axes[1].set_ylabel('Frequency')
    else:
        axes[1].text(0.5, 0.5, 'No G weights', ha='center', va='center')
    
    # B weights
    if weights_b.size > 0:
        axes[2].hist(np.abs(weights_b.flatten()), bins=30, alpha=0.7, color='orange')
        axes[2].set_title(f'Adaptive Bias Weights (n={n_b})')
        axes[2].set_xlabel('|Weight|')
        axes[2].set_ylabel('Frequency')
    else:
        axes[2].text(0.5, 0.5, 'No B weights', ha='center', va='center')
    
    plt.tight_layout()
    plt.show()

# Train model with operators
X, y = make_classification(n_samples=200, n_features=10, random_state=42)
model = GSRVFLClassifier(
    n_hidden=100,
    random_state=42,
    g_operators=[StructuredDirectLink()],
    b_operators=[AdaptiveBias()]
)
model.fit(X, y)

plot_component_weights(model)

## 5. Confusion Matrix Visualization

Visualize classification performance with confusion matrices.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Load data
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Train model
model = GSRVFLClassifier(n_hidden=200, random_state=42)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix: GS-RVFL on Digits Dataset')
plt.tight_layout()
plt.show()

## 6. Sensitivity Analysis

Visualize sensitivity to hyperparameters.

In [ ]:
def sensitivity_analysis(X_train, y_train, X_test, y_test, param_name, param_values):
    """Analyze sensitivity to a hyperparameter."""
    results = []
    
    for val in param_values:
        # Create model with parameter
        if param_name == 'lambda_reg':
            model = GSRVFLClassifier(lambda_reg=val, random_state=42)
        elif param_name == 'scale':
            model = GSRVFLClassifier(scale=val, random_state=42)
        elif param_name == 'n_hidden':
            model = GSRVFLClassifier(n_hidden=val, random_state=42)
        else:
            break
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append(acc)
    
    return results

# Test different hyperparameters
param_configs = [
    ('lambda_reg', [1e-6, 1e-4, 1e-3, 1e-2, 0.1, 1.0]),
    ('scale', [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]),
    ('n_hidden', [10, 20, 50, 100, 200, 500, 1000])
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (param_name, param_values) in zip(axes, param_configs):
    results = sensitivity_analysis(
        X_train, y_train, X_test, y_test,
        param_name, param_values
    )
    
    ax.plot(param_values, results, 'bo-', linewidth=2, markersize=8)
    ax.set_xlabel(param_name)
    ax.set_ylabel('Test Accuracy')
    ax.set_title(f'Sensitivity to {param_name}')
    ax.grid(True)
    
    if param_name in ['lambda_reg', 'scale']:
        ax.set_xscale('log')

plt.tight_layout()
plt.show()

## Summary

This visualization gallery demonstrates:

1. **Decision Boundaries**: GS-RVFL can learn complex decision boundaries
2. **Operator Impact**: Different operators affect the decision boundary shape
3. **Scalability**: Performance improves with more hidden nodes
4. **Component Importance**: Visualization of weights for different components
5. **Classification Performance**: Clear separation in confusion matrices
6. **Hyperparameter Sensitivity**: Understanding how parameters affect performance